In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
from tqdm import tqdm



# Data Preprocessing

In [2]:
# Audio settings
SAMPLE_RATE = 22050
DURATION = 30   # seconds

# Paths
INPUT_DIR = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
OUTPUT_DIR = "/kaggle/working/processed_mels"

# Stems we want to process
STEMS = ["drums.wav", "bass.wav", "vocals.wav", "other.wav"]


def preprocess_dataset():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # get all genre folders
    genres = [g for g in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, g))]

    for genre in genres:

        genre_input = os.path.join(INPUT_DIR, genre)
        genre_output = os.path.join(OUTPUT_DIR, genre)
        os.makedirs(genre_output, exist_ok=True)

        songs = os.listdir(genre_input)

        for song in tqdm(songs, desc=f"Processing {genre}"):

            song_input = os.path.join(genre_input, song)
            song_output = os.path.join(genre_output, song)

            if not os.path.isdir(song_input):
                continue

            os.makedirs(song_output, exist_ok=True)

            for stem in STEMS:

                input_file = os.path.join(song_input, stem)
                output_file = os.path.join(song_output, stem.replace(".wav", ".npy"))

                # skip if already processed
                if os.path.exists(output_file):
                    continue

                # load audio (resampled to 22050)
                audio, _ = librosa.load(input_file, sr=SAMPLE_RATE, duration=DURATION)

                # pad if audio is shorter than expected
                target_len = SAMPLE_RATE * DURATION
                if len(audio) < target_len:
                    audio = np.pad(audio, (0, target_len - len(audio)))

                # compute mel spectrogram
                mel = librosa.feature.melspectrogram(
                    y=audio,
                    sr=SAMPLE_RATE,
                    n_mels=128,
                    n_fft=2048,
                    hop_length=512
                )

                np.save(output_file, mel.astype(np.float32))


# run preprocessing
preprocess_dataset()

Processing pop: 100%|██████████| 100/100 [00:58<00:00,  1.72it/s]


In [40]:
# Audio settings
SR = 22050
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048

# Paths
INPUT_DIR = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
OUTPUT_DIR = "/kaggle/working/processed_noise_mels"


def preprocess_noise():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # get all wav files
    noise_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".wav")]

    print(f"Processing {len(noise_files)} noise files")

    for file in tqdm(noise_files):

        input_path = os.path.join(INPUT_DIR, file)
        output_path = os.path.join(OUTPUT_DIR, file.replace(".wav", ".npy"))

        # skip if already processed
        if os.path.exists(output_path):
            continue

        try:
            # load audio and resample to 22050
            audio, _ = librosa.load(input_path, sr=SR)

            # convert to mel spectrogram
            mel = librosa.feature.melspectrogram(
                y=audio,
                sr=SR,
                n_mels=N_MELS,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH
            )

            # save mel spectrogram
            np.save(output_path, mel.astype(np.float32))

        except Exception as e:
            print(f"Failed to process {file}: {e}")


# run preprocessing
preprocess_noise()

Processing 2000 noise files


100%|██████████| 2000/2000 [00:00<00:00, 147801.25it/s]


# Custom Dataset

In [50]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import librosa
import torchaudio.transforms as T


import os
import random
import numpy as np
import librosa
import torch
import torch.nn.functional as F
import torchaudio.transforms as T
from torch.utils.data import Dataset

class MashupDataset(Dataset):

    def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):

        self.processed_dir = processed_dir
        self.song_dict = song_dict
        self.genres = genres
        self.size = size
        self.is_val = is_val

        self.genre_to_idx = {g: i for i, g in enumerate(genres)}

        self.noise_files = [
            os.path.join(noise_dir, f)
            for f in os.listdir(noise_dir)
            if f.endswith(".npy")
        ]

        self.freq_mask = T.FrequencyMasking(freq_mask_param=30)
        self.time_mask = T.TimeMasking(time_mask_param=60)

        self.crop_size = 1024


    def __len__(self):
        return self.size


    def __getitem__(self, idx):

        genre = random.choice(self.genres) if not self.is_val else self.genres[idx % len(self.genres)]
        songs = self.song_dict[genre]

        target_h = 128
        target_w = 1292

        mixed = np.zeros((target_h, target_w), dtype=np.float32)

        stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

        # -------- STEM MIXING --------
        for stem in stems:

            if not self.is_val and random.random() < 0.1:
                continue

            path = os.path.join(self.processed_dir, genre, random.choice(songs), stem)

            mel = np.load(path)

            if mel.shape[1] > target_w:
                mel = mel[:, :target_w]
            else:
                mel = np.pad(mel, ((0,0),(0,target_w-mel.shape[1])))

            gain = 1.0 if self.is_val else random.uniform(0.4, 1.6)

            mixed += mel * gain


        # -------- TIME SHIFT --------
        if not self.is_val:
            shift = random.randint(-200, 200)
            mixed = np.roll(mixed, shift, axis=1)


        # -------- ADD RANDOM SILENCE --------
        if not self.is_val and random.random() < 0.3:

            silence_len = random.randint(100, 300)
            start = random.randint(0, target_w - silence_len)

            mixed[:, start:start+silence_len] = 0


        # -------- MULTI NOISE ADDITION --------
        if not self.is_val:

            for _ in range(random.randint(2,4)):

                noise = np.load(random.choice(self.noise_files))

                noise_len = noise.shape[1]

                if noise_len >= target_w:
                    noise = noise[:, :target_w]
                    start = 0
                else:
                    start = random.randint(0, target_w - noise_len)

                snr = random.uniform(8, 25)

                signal_power = mixed.mean() + 1e-8
                noise_power = noise.mean() + 1e-8

                scale = signal_power / (10**(snr/10) * noise_power)

                mixed[:, start:start+noise.shape[1]] += noise * scale


        # -------- RANDOM CROP TRAINING --------
        if not self.is_val:

            if mixed.shape[1] > self.crop_size:
                start = random.randint(0, mixed.shape[1] - self.crop_size)
                mixed = mixed[:, start:start+self.crop_size]

        else:
            mixed = mixed[:, :self.crop_size]


        # -------- LOUDNESS AUGMENTATION --------
        if not self.is_val:
            loudness = random.uniform(0.7, 1.3)
            mixed *= loudness


        # -------- MEL NORMALIZATION --------
        mel_db = librosa.power_to_db(np.maximum(mixed, 1e-10))

        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        mel = torch.tensor(mel_db).unsqueeze(0)


        # -------- SPEC AUGMENT --------
        if not self.is_val:

            mel = self.freq_mask(mel)
            mel = self.time_mask(mel)


        label = self.genre_to_idx[genre]

        return mel, label
# class MashupDataset(Dataset):

#     def __init__(self, processed_dir, noise_dir, song_dict, genres, size=30000, is_val=False):

#         self.processed_dir = processed_dir
#         self.song_dict = song_dict
#         self.genres = genres
#         self.size = size
#         self.is_val = is_val

#         self.genre_to_idx = {g: i for i, g in enumerate(genres)}

#         self.noise_files = [
#             os.path.join(noise_dir, f)
#             for f in os.listdir(noise_dir)
#             if f.endswith(".npy")
#         ]

#         # stronger specaugment
#         self.freq_mask = T.FrequencyMasking(freq_mask_param=60)
#         self.time_mask = T.TimeMasking(time_mask_param=100)


#     def __len__(self):
#         return self.size


#     def __getitem__(self, idx):

#         genre = random.choice(self.genres) if not self.is_val else self.genres[idx % len(self.genres)]

#         songs = self.song_dict[genre]

#         target_h = 128
#         target_w = 1292

#         mixed = np.zeros((target_h, target_w), dtype=np.float32)

#         stems = ["drums.npy", "bass.npy", "vocals.npy", "other.npy"]

#         # -------- STEM MIXING --------
#         for stem in stems:

#             if not self.is_val and random.random() < 0.1:
#                 # 10% chance drop stem (instrument missing)
#                 continue

#             path = os.path.join(self.processed_dir, genre, random.choice(songs), stem)
#             mel = np.load(path)

#             if mel.shape[1] > target_w:
#                 mel = mel[:, :target_w]
#             else:
#                 mel = np.pad(mel, ((0,0),(0,target_w-mel.shape[1])))

#             gain = 1.0 if self.is_val else random.uniform(0.4, 1.6)

#             mixed += mel * gain


#         # -------- TIME SHIFT --------
#         if not self.is_val:
#             shift = random.randint(-200, 200)
#             mixed = np.roll(mixed, shift, axis=1)


#         # -------- MULTI NOISE ADDITION --------
#         if not self.is_val:

#             for _ in range(random.randint(1,3)):

#                 noise = np.load(random.choice(self.noise_files))

#                 noise_len = noise.shape[1]

#                 if noise_len >= target_w:
#                     noise = noise[:, :target_w]
#                     start = 0
#                 else:
#                     start = random.randint(0, target_w - noise_len)

#                 snr = random.uniform(10, 30)

#                 signal_power = mixed.mean()
#                 noise_power = noise.mean() + 1e-8

#                 scale = signal_power / (10**(snr/10) * noise_power)

#                 mixed[:, start:start+noise.shape[1]] += noise * scale


#         # -------- MEL NORMALIZATION --------
#         mel_db = librosa.power_to_db(np.maximum(mixed, 1e-10))

#         mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

#         mel = torch.tensor(mel_db).unsqueeze(0)


#         # -------- SPECAUGMENT --------
#         if not self.is_val:

#             for _ in range(2):
#                 mel = self.freq_mask(mel)
#                 mel = self.time_mask(mel)


#         label = self.genre_to_idx[genre]

#         return mel, label

In [51]:
# 1. Stratified Split Logic
def prepare_stratified_split(root_dir, val_total=8000):
    genres = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    num_genres = len(genres)
    val_per_genre_target = val_total // num_genres
    
    train_songs = {}
    val_songs = {}

    for genre in genres:
        genre_path = os.path.join(root_dir, genre)
        # Get all subdirectories (songs)
        songs = sorted([s for s in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, s))])
        
        num_available = len(songs)
        
        # Safety Check: Don't take more than 20% of a genre for validation 
        # to ensure training always has data.
        val_limit = int(num_available * 0.2)
        actual_val_count = min(val_per_genre_target, val_limit)
        
        if actual_val_count == 0 and num_available > 0:
            actual_val_count = 1 # Ensure at least one if possible
            
        val_songs[genre] = songs[:actual_val_count]
        train_songs[genre] = songs[actual_val_count:]
        
        # Debug print to catch the empty ones
        print(f"Genre {genre}: Total={num_available}, Train={len(train_songs[genre])}, Val={len(val_songs[genre])}")
        
        if len(train_songs[genre]) == 0:
            raise ValueError(f"CRITICAL: Genre '{genre}' has 0 training songs. Decrease val_total!")

    return train_songs, val_songs, genres

In [52]:
PROCESSED_DIR = '/kaggle/working/processed_mels'
processed_noise_dir = "/kaggle/working/processed_noise_mels"

In [53]:
train_songs, val_songs, genres = prepare_stratified_split(PROCESSED_DIR, val_total=5000)

Genre blues: Total=100, Train=80, Val=20
Genre classical: Total=100, Train=80, Val=20
Genre country: Total=100, Train=80, Val=20
Genre disco: Total=100, Train=80, Val=20
Genre hiphop: Total=100, Train=80, Val=20
Genre jazz: Total=100, Train=80, Val=20
Genre metal: Total=100, Train=80, Val=20
Genre pop: Total=100, Train=80, Val=20
Genre reggae: Total=100, Train=80, Val=20
Genre rock: Total=100, Train=80, Val=20


In [54]:
train_dataset = MashupDataset(PROCESSED_DIR, processed_noise_dir, train_songs, genres, size=60000)

val_dataset = MashupDataset(PROCESSED_DIR, processed_noise_dir, val_songs, genres, size=6000, is_val=True)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

In [55]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F


# class ResidualBlock(nn.Module):

#     def __init__(self, in_ch, out_ch):
#         super().__init__()

#         self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
#         self.bn1 = nn.BatchNorm2d(out_ch)

#         self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
#         self.bn2 = nn.BatchNorm2d(out_ch)

#         self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()


#     def forward(self, x):

#         identity = self.skip(x)

#         x = F.relu(self.bn1(self.conv1(x)))
#         x = self.bn2(self.conv2(x))

#         x += identity
#         return F.relu(x)


# class AttentionPooling(nn.Module):

#     def __init__(self, input_dim):
#         super().__init__()

#         self.attn = nn.Linear(input_dim, 1)


#     def forward(self, x):

#         # x : (B,T,D)

#         weights = torch.softmax(self.attn(x), dim=1)

#         x = (weights * x).sum(dim=1)

#         return x


# class CRNN(nn.Module):

#     def __init__(self, num_classes):

#         super().__init__()

#         # CNN feature extractor
#         self.cnn = nn.Sequential(

#             ResidualBlock(1, 32),
#             nn.MaxPool2d((2,2)),
#             nn.Dropout(0.3),

#             ResidualBlock(32, 64),
#             nn.MaxPool2d((2,2)),
#             nn.Dropout(0.3),

#             ResidualBlock(64, 128),
#             nn.MaxPool2d((2,2)),
#             nn.Dropout(0.4),

#             ResidualBlock(128, 256),
#             nn.MaxPool2d((2,2)),
#             nn.Dropout(0.3),
#         )

#         # compress frequency dimension
#         self.freq_pool = nn.AdaptiveAvgPool2d((1, None))

#         # GRU
#         self.gru = nn.GRU(
#             input_size=256,
#             hidden_size=256,
#             num_layers=2,
#             bidirectional=True,
#             batch_first=True
#         )

#         # attention pooling
#         self.attention = AttentionPooling(512)

#         # classifier
#         self.fc = nn.Sequential(

#             nn.Linear(512, 256),
#             nn.ReLU(),
#             nn.Dropout(0.5),

#             nn.Linear(256, num_classes)
#         )


#     def forward(self, x):

#         x = self.cnn(x)

#         # shape (B,C,F,T)

#         x = self.freq_pool(x)

#         # now (B,C,1,T)

#         x = x.squeeze(2)

#         # (B,C,T)

#         x = x.permute(0,2,1)

#         # (B,T,C)

#         x, _ = self.gru(x)

#         x = self.attention(x)

#         x = self.fc(x)

#         return x

import torch
import torch.nn as nn
import torch.nn.functional as F


class GenreCNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        # ----- Convolution Blocks -----

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(128)

        self.conv7 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm2d(256)

        # pooling
        self.pool = nn.MaxPool2d(2,2)

        # global pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))

        # classifier
        self.fc = nn.Sequential(

            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128,num_classes)
        )


    def forward(self,x):

        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = F.relu(self.bn2(self.conv2(x)))

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)

        x = F.relu(self.bn4(self.conv4(x)))

        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool(x)

        x = F.relu(self.bn6(self.conv6(x)))

        x = F.relu(self.bn7(self.conv7(x)))

        x = self.global_pool(x)

        x = torch.flatten(x,1)

        x = self.fc(x)

        return x

In [56]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GenreCNN(num_classes=len(genres)).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.3,
    patience=2,
)

from torch.amp import GradScaler, autocast

scaler = GradScaler("cuda")

EPOCHS = 25
patience = 10

best_val_acc = 0
patience_counter = 0

In [57]:
for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ---------------- TRAIN ----------------
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(train_loader)

    for x, y in pbar:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with autocast("cuda"):

            logits = model(x)

            loss = criterion(logits, y)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()

        total += y.size(0)

        pbar.set_description(
            f"Train Loss {train_loss/(total/32):.4f} Acc {100*correct/total:.2f}%"
        )


    train_acc = correct / total


    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in tqdm(val_loader):

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == y).sum().item()

            total += y.size(0)


    val_acc = correct / total

    print(f"\nTrain Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")


    # ---------------- SCHEDULER ----------------
    scheduler.step(val_acc)


    # ---------------- EARLY STOPPING ----------------
    if val_acc > best_val_acc:

        best_val_acc = val_acc
        patience_counter = 0

        torch.save(model.state_dict(), "best_model.pth")

        print("Model Improved. Saved.")

    else:

        patience_counter += 1

        print(f"Early stopping counter: {patience_counter}/{patience}")

        if patience_counter >= patience:

            print("Early stopping triggered")
            break


print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")


Epoch 1/25


100%|██████████| 94/94 [00:16<00:00,  5.87it/s]



Train Acc: 0.7214
Val Acc: 0.6298
Model Improved. Saved.

Epoch 2/25


100%|██████████| 94/94 [00:15<00:00,  5.88it/s]



Train Acc: 0.8771
Val Acc: 0.5258
Early stopping counter: 1/10

Epoch 3/25


...


Train Acc: 0.9902
Val Acc: 0.8682
Early stopping counter: 4/10

Epoch 17/25


100%|██████████| 94/94 [00:16<00:00,  5.79it/s]



Train Acc: 0.9903
Val Acc: 0.8790
Model Improved. Saved.

Epoch 18/25


Train Loss 0.0450 Acc 98.44%:   0%|          | 4/1875 [00:00<07:41,  4.05it/s] 


KeyboardInterrupt: 

In [58]:
import librosa
import numpy as np

SR = 22050
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
TARGET_WIDTH = 1292


def audio_to_mel(path):

    audio, _ = librosa.load(path, sr=SR)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    # crop / pad to fixed width
    if mel.shape[1] > TARGET_WIDTH:
        mel = mel[:, :TARGET_WIDTH]
    else:
        mel = np.pad(mel, ((0,0),(0, TARGET_WIDTH - mel.shape[1])))

    mel = librosa.power_to_db(np.maximum(mel, 1e-10))

    # normalize
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel.astype(np.float32)

In [59]:
import pandas as pd
import torch
from torch.utils.data import Dataset
import os


class MashupTestDataset(Dataset):

    def __init__(self, test_csv, mashup_dir):

        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = str(row["id"]).zfill(4)

        filename = row["filename"] if "filename" in row else sample_id + ".wav"

        path = os.path.join(self.mashup_dir, filename)

        mel = audio_to_mel(path)

        mel = torch.tensor(mel).unsqueeze(0)

        return mel, sample_id

In [60]:
test_csv = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

test_dataset = MashupTestDataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4   
)

In [61]:
genres = [
    "blues","classical","country","disco","hiphop",
    "jazz","metal","pop","reggae","rock"
]

predictions = []

model.eval()

with torch.no_grad():

    for mel, ids in tqdm(test_loader, desc="Predicting"):

        mel = mel.to(device)

        outputs = model(mel)

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        for sample_id, pred in zip(ids, preds):
            predictions.append((sample_id, genres[pred]))

Predicting: 100%|██████████| 189/189 [02:28<00:00,  1.27it/s]


In [63]:
import pandas as pd

submission = pd.DataFrame(predictions, columns=["id", "genre"])

submission.to_csv("submission.csv", index=False)

print(submission.head())

     id      genre
0  0001        pop
1  0002  classical
2  0003      disco
3  0004      metal
4  0005    country


In [37]:
submission

,id,genre
0,0001,pop
1,0002,blues
2,0003,disco
3,0004,metal
4,0005,blues
...,...,...
3015,3016,rock
3016,3017,blues
3017,3018,reggae
3018,3019,rock
